# Extract Central Bankers From Wikipedia

Goal: build a clean table with one row per person and keep the country.

Why we need two Wikipedia pages:
- `List of central banks`: gives us the mapping `central bank -> country`
- `Category:Central bankers`: gives us categories such as `Governors of the Bank of Albania`, where we can extract the people names

At the end, we join both ideas:
- the category tells us the bank and the role
- the bank list lets us recover the country

## 1. Load Packages

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CentralBankNotebook/1.0)"}

## 2. Download the list of central banks

This cell creates `central_banks_df`.

We only keep two columns because that is all we need here:
- `country`
- `central_bank`

This table is important because later we will use it to recover the country from the bank name.

In [ ]:
WIKI_BANKS_URL = "https://en.wikipedia.org/wiki/List_of_central_banks"

response = requests.get(WIKI_BANKS_URL, headers=HEADERS, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
table = soup.select_one("table.wikitable")

records = []
for row in table.select("tr")[1:]:
    cols = row.find_all(["th", "td"])
    if len(cols) < 3:
        continue

    records.append(
        {
            "country": cols[0].get_text(" ", strip=True),
            "central_bank": cols[2].get_text(" ", strip=True),
        }
    )

central_banks_df = pd.DataFrame(records)
central_banks_df.head(10)

## 3. Download the central bankers categories

This cell creates `central_bankers_categories_df`.

Example categories:
- `Governors of the Bank of Albania`
- `Presidents of the Central Bank of Argentina`

We keep only categories that look like roles we care about: governor, president, chair.

In [ ]:
CATEGORY_URL = "https://en.wikipedia.org/wiki/Category:Central_bankers"

category_response = requests.get(CATEGORY_URL, headers=HEADERS, timeout=30)
category_response.raise_for_status()

category_soup = BeautifulSoup(category_response.text, "html.parser")
category_section = category_soup.select_one("#mw-subcategories")

category_rows = []
if category_section:
    for link in category_section.select("a[href]"):
        href = link.get("href", "")
        if not href.startswith("/wiki/Category:"):
            continue

        category_rows.append(
            {
                "category_name": link.get_text(" ", strip=True),
                "category_url": "https://en.wikipedia.org" + href,
            }
        )

central_bankers_categories_df = pd.DataFrame(category_rows).drop_duplicates()
central_bankers_categories_df = central_bankers_categories_df[
    central_bankers_categories_df["category_name"].str.contains(
        r"governors?|presidents?|chair(men|women|persons?)?",
        case=False,
        na=False,
    )
].reset_index(drop=True)

central_bankers_categories_df.head(20)

## 4. Extract the names inside each category

This cell adds two columns to `central_bankers_categories_df`:
- `governor_names`: list of names found in that category
- `governor_count`: how many names were found

This is still an intermediate step. The names are still grouped as a list inside each category.

In [ ]:
def extract_people_from_category(category_url):
    response = requests.get(category_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    pages_section = soup.select_one("#mw-pages")

    names = []
    if pages_section:
        for item in pages_section.select("div.mw-category-group li"):
            link = item.find("a", href=True)
            if not link:
                continue

            href = link.get("href", "")
            if not href.startswith("/wiki/") or href.startswith("/wiki/Category:"):
                continue

            names.append(link.get_text(" ", strip=True))

    return sorted(set(names))

central_bankers_categories_df = central_bankers_categories_df.copy()
central_bankers_categories_df["governor_names"] = central_bankers_categories_df["category_url"].apply(extract_people_from_category)
central_bankers_categories_df["governor_count"] = central_bankers_categories_df["governor_names"].apply(len)

central_bankers_categories_df.head(10)

## 5. Expand to one row per person and recover the country

This is the final clean step.

What happens here:
1. `explode(governor_names)` turns one list into many rows
2. we infer the role from the category name
3. we infer the bank name from the category name
4. we match that bank name against `central_banks_df` to recover `country`
5. we save the clean file `governors_clean_names.csv`

In [ ]:
df = central_bankers_categories_df.copy()
df = df.explode("governor_names").reset_index(drop=True)
df = df.rename(columns={"governor_names": "PName_original"})
df = df[df["PName_original"].notna()].copy()

df["PName_original"] = df["PName_original"].astype(str).str.strip()
df = df[df["PName_original"] != ""]
df = df[df["PName_original"].str.split().str.len() >= 2]

def infer_position(category_name):
    text = str(category_name).lower()
    if "governor" in text:
        return "Governor"
    if "president" in text:
        return "President"
    if "chair" in text:
        return "Chair"
    return ""

def infer_bank_name(category_name):
    text = str(category_name)
    prefixes = [
        "Governors of ",
        "Presidents of ",
        "Chairmen of ",
        "Chairwomen of ",
        "Chairpersons of ",
        "Chairs of ",
    ]
    for prefix in prefixes:
        if text.startswith(prefix):
            return text.replace(prefix, "").strip()
    return text.strip()

bank_lookup_df = central_banks_df[["country", "central_bank"]].drop_duplicates().copy()
bank_lookup_df["central_bank_norm"] = bank_lookup_df["central_bank"].str.lower().str.strip()
bank_to_country = dict(zip(bank_lookup_df["central_bank_norm"], bank_lookup_df["country"]))

df["central_bank_name"] = df["category_name"].apply(infer_bank_name)
df["country"] = df["central_bank_name"].str.lower().str.strip().map(bank_to_country).fillna("")
df["PName"] = df["PName_original"]
df[["first", "last"]] = df["PName"].str.split(" ", n=1, expand=True)
df["last"] = df["last"].fillna("")
df["Position"] = df["category_name"].apply(infer_position)

df = df.drop_duplicates(subset=["country", "central_bank_name", "PName_original", "Position"])
df = df[["country", "central_bank_name", "PName_original", "PName", "first", "last", "Position", "category_name", "category_url"]]
df = df.sort_values(["country", "central_bank_name", "PName_original"]).reset_index(drop=True)

df.to_csv("governors_clean_names.csv", index=False)
print(df.head(20))
print("\nTotal names:", len(df))